In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path(r"C:\project 3\Paper1_SSL")

search_dirs = [
    PROJECT / "03_outputs" / "final_results_lock",
    PROJECT / "03_outputs" / "external_validation",
    PROJECT / "03_outputs" / "models" / "PBMMAE"
]

keywords = [
    "physical",
    "signature",
    "domain",
    "character",
    "effect",
    "epsilon",
    "gpi",
    "external"
]

print("Candidate files for Fig. 4:\n")

for folder in search_dirs:
    if folder.exists():
        for p in folder.rglob("*"):
            if p.is_file():
                name = p.name.lower()

                if any(k in name for k in keywords):
                    print(p)

In [ ]:
# =========================================================
# Load authoritative physical-characterization files
# =========================================================

PHYS_DIR = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
    / "physical_characterization"
)

SIGNATURE_FILE = PHYS_DIR / "PBMMAE_4domain_robust_standardized_signatures.csv"

EFFECT_FILE = PHYS_DIR / "PBMMAE_4domain_channel_separation_kruskal_effectsize.csv"

signatures = pd.read_csv(SIGNATURE_FILE)
effects = pd.read_csv(EFFECT_FILE)

print("SIGNATURE FILE")
print("=" * 70)
print("Shape:", signatures.shape)
print("Columns:")
print(signatures.columns.tolist())
print("\nFirst rows:")
display(signatures.head(20))

print("\n\nEFFECT-SIZE FILE")
print("=" * 70)
print("Shape:", effects.shape)
print("Columns:")
print(effects.columns.tolist())
print("\nFirst rows:")
display(effects.head(20))

In [ ]:
# =========================================================
# Fig. 4(a): Robust standardized physical signatures
# =========================================================

# Preserve the locked 14-channel order used in the model
channel_order = [
    "RTE_TMI",
    "CBG",
    "CBG_RES",
    "HGM",
    "K",
    "eTh",
    "eU",
    "CPD",
    "MAG_LD",
    "GRAV_LD",
    "DEM_LD",
    "ID",
    "DEM",
    "SLOPE"
]

# Convert long table -> 4 x 14 matrix
signature_matrix = (
    signatures
    .pivot(
        index="Meta_domain",
        columns="Channel",
        values="Robust_standardized_median"
    )
    .reindex(
        index=[0, 1, 2, 3],
        columns=channel_order
    )
)

# Rename rows for figure
signature_matrix.index = ["M0", "M1", "M2", "M3"]

print("Physical-signature matrix:")
display(signature_matrix.round(3))

In [ ]:
# =========================================================
# Fig. 4(a): Robust standardized physical signatures
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# File path
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

SIGNATURE_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
    / "physical_characterization"
    / "PBMMAE_4domain_robust_standardized_signatures.csv"
)

print("Signature file exists:", SIGNATURE_FILE.exists())
print("Signature file:", SIGNATURE_FILE)

# ---------------------------------------------------------
# Load signature table
# ---------------------------------------------------------
signature_df = pd.read_csv(SIGNATURE_FILE)

print("\nShape:", signature_df.shape)
print("Columns:")
print(signature_df.columns.tolist())

# ---------------------------------------------------------
# Channel order (fixed)
# ---------------------------------------------------------
channel_order = [
    "RTE_TMI", "CBG", "CBG_RES", "HGM",
    "K", "eTh", "eU",
    "CPD",
    "MAG_LD", "GRAV_LD", "DEM_LD", "ID",
    "DEM", "SLOPE"
]

domain_order = [0, 1, 2, 3]
domain_labels = ["M0", "M1", "M2", "M3"]

# ---------------------------------------------------------
# Pivot to heatmap matrix
# ---------------------------------------------------------
heatmap_df = (
    signature_df
    .pivot(index="Meta_domain", columns="Channel", values="Robust_standardized_median")
    .reindex(index=domain_order, columns=channel_order)
)

print("\nHeatmap matrix:")
display(heatmap_df)

data = heatmap_df.values

# ---------------------------------------------------------
# Plot
# ---------------------------------------------------------
fig, ax = plt.subplots(figsize=(10.5, 4.8))

vmax = np.nanmax(np.abs(data))

im = ax.imshow(
    data,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    aspect="auto"
)

# ---------------------------------------------------------
# Tick labels
# ---------------------------------------------------------
ax.set_xticks(np.arange(len(channel_order)))
ax.set_xticklabels(channel_order, rotation=45, ha="right",
                   fontsize=10, fontweight="bold")

ax.set_yticks(np.arange(len(domain_labels)))
ax.set_yticklabels(domain_labels, fontsize=11, fontweight="bold")

# ---------------------------------------------------------
# Axis labels
# ---------------------------------------------------------
ax.set_ylabel("PB-MMAE domain", fontsize=12, fontweight="bold")
ax.set_xlabel("Physical variable", fontsize=12, fontweight="bold")

# ---------------------------------------------------------
# Annotate cell values
# ---------------------------------------------------------
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        value = data[i, j]
        ax.text(
            j, i, f"{value:.2f}",
            ha="center", va="center",
            fontsize=8, fontweight="bold", color="black"
        )

# ---------------------------------------------------------
# Add separators between physics families
# ---------------------------------------------------------
# Families:
# Magnetic | Gravity | Radiometric | Thermal | Structural | Terrain

family_breaks = [
    0.5,   # after RTE_TMI
    3.5,   # after HGM
    6.5,   # after eU
    7.5,   # after CPD
    11.5   # after ID
]

for x in family_breaks:
    ax.axvline(
        x,
        color="black",
        linewidth=0.8,
        alpha=0.55
    )

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.01, 0.98, "(a)",
    transform=ax.transAxes,
    fontsize=13, fontweight="bold",
    ha="left", va="top"
)

# ---------------------------------------------------------
# Colorbar
# ---------------------------------------------------------
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Robust standardized median", fontsize=11, fontweight="bold")
cbar.ax.tick_params(labelsize=10)

# ---------------------------------------------------------
# Frame styling
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# FIG. 4(a) — ROBUST STANDARDIZED PHYSICAL SIGNATURES
# Very-large-text publication version
# =========================================================

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# File path
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

SIGNATURE_FILE = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
    / "physical_characterization"
    / "PBMMAE_4domain_robust_standardized_signatures.csv"
)

print("Signature file exists:", SIGNATURE_FILE.exists())

# ---------------------------------------------------------
# Load data
# ---------------------------------------------------------
signature_df = pd.read_csv(SIGNATURE_FILE)

channel_order = [
    "RTE_TMI", "CBG", "CBG_RES", "HGM",
    "K", "eTh", "eU",
    "CPD",
    "MAG_LD", "GRAV_LD", "DEM_LD", "ID",
    "DEM", "SLOPE"
]

domain_order = [0, 1, 2, 3]
domain_labels = ["M0", "M1", "M2", "M3"]

heatmap_df = (
    signature_df
    .pivot(
        index="Meta_domain",
        columns="Channel",
        values="Robust_standardized_median"
    )
    .reindex(index=domain_order, columns=channel_order)
)

data = heatmap_df.values

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(19.0, 8.8))

vmax = np.nanmax(np.abs(data))

im = ax.imshow(
    data,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    aspect="auto",
    interpolation="nearest"
)

# ---------------------------------------------------------
# Large tick labels
# ---------------------------------------------------------
ax.set_xticks(np.arange(len(channel_order)))
ax.set_xticklabels(
    channel_order,
    rotation=38,
    ha="right",
    fontsize=22,
    fontweight="bold"
)

ax.set_yticks(np.arange(len(domain_labels)))
ax.set_yticklabels(
    domain_labels,
    fontsize=25,
    fontweight="bold"
)

# ---------------------------------------------------------
# Large axis labels
# ---------------------------------------------------------
ax.set_ylabel(
    "PB-MMAE domain",
    fontsize=29,
    fontweight="bold"
)

ax.set_xlabel(
    "Physical variable",
    fontsize=29,
    fontweight="bold"
)

# ---------------------------------------------------------
# Large cell values with adaptive contrast
# ---------------------------------------------------------
for i in range(data.shape[0]):
    for j in range(data.shape[1]):

        value = data[i, j]

        if np.isfinite(value):
            rgba = im.cmap(im.norm(value))
            luminance = (
                0.299 * rgba[0]
                + 0.587 * rgba[1]
                + 0.114 * rgba[2]
            )

            text_color = "white" if luminance < 0.55 else "black"

            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=19,
                fontweight="bold",
                color=text_color
            )

# ---------------------------------------------------------
# Physics-family separators
# Magnetic | Gravity | Radiometric | Thermal | Structural | Terrain
# ---------------------------------------------------------
family_breaks = [
    0.5,
    3.5,
    6.5,
    7.5,
    11.5
]

for x in family_breaks:
    ax.axvline(
        x,
        color="black",
        linewidth=1.8,
        alpha=0.68
    )

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.01,
    0.98,
    "(a)",
    transform=ax.transAxes,
    fontsize=34,
    fontweight="bold",
    ha="left",
    va="top",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.84,
        pad=1.2
    )
)

# ---------------------------------------------------------
# Large colorbar
# ---------------------------------------------------------
cbar = plt.colorbar(
    im,
    ax=ax,
    fraction=0.035,
    pad=0.03
)

cbar.set_label(
    "Robust standardized median",
    fontsize=24,
    fontweight="bold"
)

cbar.ax.tick_params(
    labelsize=20,
    width=1.3,
    length=6
)

for tick in cbar.ax.get_yticklabels():
    tick.set_fontweight("bold")

# ---------------------------------------------------------
# Frame
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.4)

plt.tight_layout()
# ---------------------------------------------------------
# Save Figure 4(a)
# ---------------------------------------------------------
OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "Fig04_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

png_path = (
    OUTPUT_DIR
    / "Paper1_Fig04a_Robust_Standardized_Physical_Signatures_600dpi.png"
)

tif_path = (
    OUTPUT_DIR
    / "Paper1_Fig04a_Robust_Standardized_Physical_Signatures_600dpi.tif"
)

fig.savefig(
    png_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig.savefig(
    tif_path,
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# Fig. 4(b): Domain-separation effect sizes
# Final styling — panel label inside axes
# =========================================================

# Sort strongest at top
effects_plot = (
    effects
    .sort_values("Epsilon_squared", ascending=True)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.5, 5.8))

bars = ax.barh(
    effects_plot["Channel"],
    effects_plot["Epsilon_squared"]
)

# ---------------------------------------------------------
# Annotate values
# ---------------------------------------------------------
for bar, value in zip(
    bars,
    effects_plot["Epsilon_squared"]
):
    ax.text(
        value + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# ---------------------------------------------------------
# Panel label — inside axes, consistent with panel (a)
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Axes
# ---------------------------------------------------------
ax.set_xlabel(
    r"Domain-separation effect size ($\epsilon^2$)",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel("")

ax.tick_params(
    axis="both",
    labelsize=11,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.set_xlim(
    0,
    effects_plot["Epsilon_squared"].max() + 0.055
)

# ---------------------------------------------------------
# Light grid
# ---------------------------------------------------------
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.22
)

# ---------------------------------------------------------
# Frame styling
# ---------------------------------------------------------
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# FIG. 4(b) — DOMAIN-SEPARATION EFFECT SIZES
# Standalone, extra-large-text publication panel
# =========================================================

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")

PHYS_DIR = (
    PROJECT
    / "03_outputs"
    / "models"
    / "PBMMAE"
    / "final_4domain_consensus"
    / "physical_characterization"
)

EFFECT_FILE = (
    PHYS_DIR
    / "PBMMAE_4domain_channel_separation_kruskal_effectsize.csv"
)

OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "Fig04_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load effect-size data
# ---------------------------------------------------------
print("Effect-size file exists:", EFFECT_FILE.exists())
print(EFFECT_FILE)

effects = pd.read_csv(EFFECT_FILE)

required_columns = {"Channel", "Epsilon_squared"}
missing_columns = required_columns - set(effects.columns)

if missing_columns:
    raise ValueError(
        f"Missing required column(s): {missing_columns}\n"
        f"Available columns: {effects.columns.tolist()}"
    )

effects["Epsilon_squared"] = pd.to_numeric(
    effects["Epsilon_squared"],
    errors="coerce"
)

effects = effects.dropna(
    subset=["Channel", "Epsilon_squared"]
).copy()

# Weakest at bottom; strongest at top
effects_plot = (
    effects
    .sort_values("Epsilon_squared", ascending=True)
    .reset_index(drop=True)
)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(13.0, 9.8))

bars = ax.barh(
    effects_plot["Channel"],
    effects_plot["Epsilon_squared"],
    zorder=3
)

# ---------------------------------------------------------
# Value labels
# ---------------------------------------------------------
for bar, value in zip(bars, effects_plot["Epsilon_squared"]):
    ax.text(
        value + 0.006,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        ha="left",
        fontsize=20,
        fontweight="bold",
        zorder=5
    )

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=34,
    fontweight="bold",
    va="top",
    ha="left",
    bbox=dict(
        facecolor="white",
        edgecolor="none",
        alpha=0.88,
        pad=1.2
    ),
    zorder=6
)

# ---------------------------------------------------------
# Axes
# \varepsilon² matches the Epsilon_squared data column
# ---------------------------------------------------------
ax.set_xlabel(
    r"Domain-separation effect size ($\eta_{H}^{2}$)",
    fontsize=28,
    fontweight="bold",
    labelpad=12
)
ax.set_ylabel("")

ax.tick_params(
    axis="both",
    labelsize=22,
    width=1.5,
    length=7
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.set_xlim(
    0,
    effects_plot["Epsilon_squared"].max() + 0.10
)

# ---------------------------------------------------------
# Styling
# ---------------------------------------------------------
ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.22,
    zorder=0
)

for spine in ax.spines.values():
    spine.set_linewidth(1.3)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = OUTPUT_DIR / "Paper1_Fig04b_Domain_Separation_Effect_Sizes_600dpi.png"
tif_path = OUTPUT_DIR / "Paper1_Fig04b_Domain_Separation_Effect_Sizes_600dpi.tif"

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("\nSaved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# Fig. 4(c): Refined post hoc external-evidence summary
# =========================================================

import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(8.2, 3.9))

ax.set_xlim(0, 3)
ax.set_ylim(0, 1)
ax.axis("off")

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.015,
    0.97,
    "(c)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Box geometry
# ---------------------------------------------------------
box_x = [0.15, 1.08, 2.01]
box_w = 0.78
box_h = 0.48
box_y = 0.24

box_face = "#F2F2F2"
box_edge = "black"

# ---------------------------------------------------------
# Helper function
# ---------------------------------------------------------
def add_evidence_box(x, title, statistic, note):
    box = FancyBboxPatch(
        (x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.03",
        linewidth=1.2,
        edgecolor=box_edge,
        facecolor=box_face
    )
    ax.add_patch(box)

    # Title
    ax.text(
        x + box_w/2,
        box_y + 0.37,
        title,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

    # Main statistic
    ax.text(
        x + box_w/2,
        box_y + 0.23,
        statistic,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    # Supporting note
    ax.text(
        x + box_w/2,
        box_y + 0.10,
        note,
        ha="center",
        va="center",
        fontsize=10.5
    )

# ---------------------------------------------------------
# Geology
# ---------------------------------------------------------
add_evidence_box(
    box_x[0],
    "Geology",
    r"Cramér's $V$ = 0.395",
    r"$p_{\mathrm{perm}} < 0.001$"
)

# ---------------------------------------------------------
# AHP-GPI
# ---------------------------------------------------------
add_evidence_box(
    box_x[1],
    "AHP-GPI",
    r"$\epsilon^2$ = 0.144",
    r"M0 $\approx$ M1 > M2 > M3"
)

# ---------------------------------------------------------
# Warm springs
# ---------------------------------------------------------
add_evidence_box(
    box_x[2],
    "Warm springs",
    "5/5 resolved springs in M2",
    r"Spatial $p$ = 0.173"
)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# FIG. 4(c) — POST HOC EXTERNAL-EVIDENCE SUMMARY
# Extra-large-text publication panel
# =========================================================

from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch

# ---------------------------------------------------------
# Output folder
# ---------------------------------------------------------
PROJECT = Path(r"C:\project 3\Paper1_SSL")
OUTPUT_DIR = PROJECT / "03_outputs" / "figures" / "Fig04_panels"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# =========================================================
# Plot
# =========================================================
fig, ax = plt.subplots(figsize=(15.5, 6.6))

ax.set_xlim(0, 3)
ax.set_ylim(0, 1)
ax.axis("off")

# ---------------------------------------------------------
# Panel label
# ---------------------------------------------------------
ax.text(
    0.02,
    0.95,
    "(c)",
    transform=ax.transAxes,
    fontsize=34,
    fontweight="bold",
    va="top",
    ha="left"
)

# ---------------------------------------------------------
# Box geometry
# ---------------------------------------------------------
box_x = [0.12, 1.08, 2.04]
box_w = 0.84
box_h = 0.50
box_y = 0.23

box_face = "#F2F2F2"
box_edge = "black"

# ---------------------------------------------------------
# Helper function
# ---------------------------------------------------------
def add_evidence_box(x, title, statistic, note):
    box = FancyBboxPatch(
        (x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.03",
        linewidth=2.0,
        edgecolor=box_edge,
        facecolor=box_face
    )
    ax.add_patch(box)

    # Heading
    ax.text(
        x + box_w / 2,
        box_y + 0.365,
        title,
        ha="center",
        va="center",
        fontsize=26,
        fontweight="bold"
    )

    # Main result
    ax.text(
        x + box_w / 2,
        box_y + 0.225,
        statistic,
        ha="center",
        va="center",
        fontsize=22,
        fontweight="bold"
    )

    # Supporting result
    ax.text(
        x + box_w / 2,
        box_y + 0.095,
        note,
        ha="center",
        va="center",
        fontsize=21,
        fontweight="normal"
    )

# ---------------------------------------------------------
# Geology
# ---------------------------------------------------------
add_evidence_box(
    box_x[0],
    "Geology",
    r"Cramér's $V$ = 0.395",
    r"$p_{\mathrm{perm}} < 0.001$"
)

# ---------------------------------------------------------
# Geothermal prospectivity index
# ---------------------------------------------------------
add_evidence_box(
    box_x[1],
    "GPI",
    r"$\eta_{H}^{2}$ = 0.144",
    r"M0 $\approx$ M1 > M2 > M3"
)

# ---------------------------------------------------------
# Warm springs
# ---------------------------------------------------------
add_evidence_box(
    box_x[2],
    "Warm springs",
    "5/5 resolved springs\nin M2",
    r"Spatial $p$ = 0.173"
)

plt.tight_layout()

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------
png_path = OUTPUT_DIR / "Paper1_Fig04c_External_Evidence_Summary_600dpi.png"
tif_path = OUTPUT_DIR / "Paper1_Fig04c_External_Evidence_Summary_600dpi.tif"

fig.savefig(png_path, dpi=600, bbox_inches="tight", facecolor="white")
fig.savefig(tif_path, dpi=600, bbox_inches="tight", facecolor="white")

plt.show()

print("Saved:")
print(png_path)
print(tif_path)

In [ ]:
# =========================================================
# EXPORT FIG. 4 PANELS (a), (b), AND (c) INDIVIDUALLY
# 600 dpi PNG + TIFF
# =========================================================

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import FancyBboxPatch

# ---------------------------------------------------------
# Output folder
# ---------------------------------------------------------
FIG04_DIR = (
    PROJECT
    / "03_outputs"
    / "figures"
    / "Fig04_panels"
)

FIG04_DIR.mkdir(parents=True, exist_ok=True)

print("Saving Fig. 4 panels to:")
print(FIG04_DIR)


# =========================================================
# PANEL (a): PHYSICAL-SIGNATURE HEATMAP
# =========================================================

fig_a, ax = plt.subplots(figsize=(10.5, 4.8))

data = heatmap_df.values
vmax = np.nanmax(np.abs(data))

im = ax.imshow(
    data,
    cmap="RdBu_r",
    vmin=-vmax,
    vmax=vmax,
    aspect="auto",
    interpolation="nearest"
)

ax.set_xticks(np.arange(len(channel_order)))
ax.set_xticklabels(
    channel_order,
    rotation=45,
    ha="right",
    fontsize=10,
    fontweight="bold"
)

ax.set_yticks(np.arange(len(domain_labels)))
ax.set_yticklabels(
    domain_labels,
    fontsize=11,
    fontweight="bold"
)

ax.set_ylabel(
    "PB-MMAE domain",
    fontsize=12,
    fontweight="bold"
)

ax.set_xlabel(
    "Physical variable",
    fontsize=12,
    fontweight="bold"
)

# Cell values
for i in range(data.shape[0]):
    for j in range(data.shape[1]):
        value = data[i, j]

        txt_color = (
            "white"
            if abs(value) > 0.55 * vmax
            else "black"
        )

        ax.text(
            j,
            i,
            f"{value:.2f}",
            ha="center",
            va="center",
            fontsize=8,
            fontweight="bold",
            color=txt_color
        )

# Physics-family separators
family_breaks = [
    0.5,   # Magnetic | Gravity
    3.5,   # Gravity | Radiometric
    6.5,   # Radiometric | Thermal
    7.5,   # Thermal | Structural
    11.5   # Structural | Terrain
]

for x in family_breaks:
    ax.axvline(
        x,
        color="black",
        linewidth=0.8,
        alpha=0.55
    )

# Panel label
ax.text(
    0.01,
    0.98,
    "(a)",
    transform=ax.transAxes,
    fontsize=13,
    fontweight="bold",
    ha="left",
    va="top"
)

# Colorbar
cbar = plt.colorbar(
    im,
    ax=ax,
    fraction=0.03,
    pad=0.02
)

cbar.set_label(
    "Robust standardized median",
    fontsize=11,
    fontweight="bold"
)

cbar.ax.tick_params(labelsize=10)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

fig_a.savefig(
    FIG04_DIR / "Paper1_Fig04a_Physical_Signatures_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig_a.savefig(
    FIG04_DIR / "Paper1_Fig04a_Physical_Signatures_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig_a)

print("Saved panel (a)")


# =========================================================
# PANEL (b): EFFECT-SIZE RANKING
# =========================================================

effects_plot = (
    effects
    .sort_values("Epsilon_squared", ascending=True)
    .reset_index(drop=True)
)

fig_b, ax = plt.subplots(figsize=(7.5, 5.8))

bars = ax.barh(
    effects_plot["Channel"],
    effects_plot["Epsilon_squared"]
)

# Value labels
for bar, value in zip(
    bars,
    effects_plot["Epsilon_squared"]
):
    ax.text(
        value + 0.005,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

# Panel label
ax.text(
    0.015,
    0.985,
    "(b)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

ax.set_xlabel(
    r"Domain-separation effect size ($\epsilon^2$)",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel("")

ax.tick_params(
    axis="both",
    labelsize=11,
    width=1.2,
    length=5
)

for tick in ax.get_xticklabels() + ax.get_yticklabels():
    tick.set_fontweight("bold")

ax.set_xlim(
    0,
    effects_plot["Epsilon_squared"].max() + 0.055
)

ax.grid(
    axis="x",
    linestyle="--",
    alpha=0.22
)

for spine in ax.spines.values():
    spine.set_linewidth(1.0)

plt.tight_layout()

fig_b.savefig(
    FIG04_DIR / "Paper1_Fig04b_Effect_Size_Ranking_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig_b.savefig(
    FIG04_DIR / "Paper1_Fig04b_Effect_Size_Ranking_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig_b)

print("Saved panel (b)")


# =========================================================
# PANEL (c): EXTERNAL-EVIDENCE SUMMARY
# =========================================================

fig_c, ax = plt.subplots(figsize=(8.2, 3.9))

ax.set_xlim(0, 3)
ax.set_ylim(0, 1)
ax.axis("off")

# Panel label
ax.text(
    0.015,
    0.97,
    "(c)",
    transform=ax.transAxes,
    fontsize=14,
    fontweight="bold",
    va="top",
    ha="left"
)

# Box geometry
box_x = [0.15, 1.08, 2.01]
box_w = 0.78
box_h = 0.48
box_y = 0.24

box_face = "#F2F2F2"
box_edge = "black"

def add_evidence_box(x, title, statistic, note):

    box = FancyBboxPatch(
        (x, box_y),
        box_w,
        box_h,
        boxstyle="round,pad=0.03",
        linewidth=1.2,
        edgecolor=box_edge,
        facecolor=box_face
    )

    ax.add_patch(box)

    ax.text(
        x + box_w / 2,
        box_y + 0.37,
        title,
        ha="center",
        va="center",
        fontsize=12,
        fontweight="bold"
    )

    ax.text(
        x + box_w / 2,
        box_y + 0.23,
        statistic,
        ha="center",
        va="center",
        fontsize=11,
        fontweight="bold"
    )

    ax.text(
        x + box_w / 2,
        box_y + 0.10,
        note,
        ha="center",
        va="center",
        fontsize=10.5
    )

# Geology
add_evidence_box(
    box_x[0],
    "Geology",
    r"Cramér's $V$ = 0.395",
    r"$p_{\mathrm{perm}} < 0.001$"
)

# AHP-GPI
add_evidence_box(
    box_x[1],
    "AHP-GPI",
    r"$\epsilon^2$ = 0.144",
    r"M0 $\approx$ M1 > M2 > M3"
)

# Warm springs
add_evidence_box(
    box_x[2],
    "Warm springs",
    "5/5 resolved springs in M2",
    r"Spatial $p$ = 0.173"
)

plt.tight_layout()

fig_c.savefig(
    FIG04_DIR / "Paper1_Fig04c_External_Evidence_Summary_600dpi.png",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

fig_c.savefig(
    FIG04_DIR / "Paper1_Fig04c_External_Evidence_Summary_600dpi.tif",
    dpi=600,
    bbox_inches="tight",
    facecolor="white"
)

plt.close(fig_c)

print("Saved panel (c)")


# =========================================================
# CONFIRM OUTPUT FILES
# =========================================================

print("\nSaved Fig. 4 files:\n")

for p in sorted(FIG04_DIR.iterdir()):
    print(p.name)